<a href="https://colab.research.google.com/github/NimaTamang/lang-translator/blob/feature/translator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import shutil
import os

# Safely delete the accidental nested duplicate folder
ghost_folder = "/content/lang-translator/lang-translator"
if os.path.exists(ghost_folder):
    shutil.rmtree(ghost_folder)
    print("Ghost folder successfully deleted. System is 100% clean!")
else:
    print("No ghost folder found. You are good to go.")


Ghost folder successfully deleted. System is 100% clean!


In [ ]:
from __future__ import annotations

In [ ]:
from pathlib import Path

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
from src.model.decoder import ZanxDecoder
from src.model.encoder import EnglishEncoder

In [ ]:
class EnToZanxTranslator(nn.Module):
    def __init__(
        self,
        encoder_path: str | Path,
        vocab_size: int,
        *,
        freeze_encoder: bool = True,
        decoder_layers: int = 2,
        decoder_dim: int = 256,
        decoder_heads: int = 4,
        max_len: int = 128,
    ) -> None:
        super().__init__()
        self.encoder = EnglishEncoder(encoder_path, freeze=freeze_encoder)
        if self.encoder.hidden_size != decoder_dim:
            self.encoder_proj = nn.Linear(self.encoder.hidden_size, decoder_dim)
        else:
            self.encoder_proj = nn.Identity()
        self.decoder = ZanxDecoder(
            vocab_size,
            d_model=decoder_dim,
            nhead=decoder_heads,
            num_layers=decoder_layers,
            max_len=max_len,
        )
        self.pad_id = 0

    def encode(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        memory = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        memory = self.encoder_proj(memory)
        memory_padding = attention_mask == 0
        return memory, memory_padding

    def forward(
        self,
        en_input_ids: torch.Tensor,
        en_attention_mask: torch.Tensor,
        zanx_input_ids: torch.Tensor,
    ) -> torch.Tensor:
        memory, memory_padding = self.encode(en_input_ids, en_attention_mask)
        tgt_padding = zanx_input_ids == self.pad_id
        return self.decoder(
            zanx_input_ids,
            memory,
            tgt_key_padding_mask=tgt_padding,
            memory_key_padding_mask=memory_padding,
        )

    def compute_loss(
        self,
        en_input_ids: torch.Tensor,
        en_attention_mask: torch.Tensor,
        zanx_input_ids: torch.Tensor,
    ) -> torch.Tensor:
        decoder_input = zanx_input_ids[:, :-1]
        targets = zanx_input_ids[:, 1:]
        logits = self.forward(en_input_ids, en_attention_mask, decoder_input)
        return F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            ignore_index=self.pad_id,
        )

    @torch.no_grad()
    def translate(
        self,
        en_input_ids: torch.Tensor,
        en_attention_mask: torch.Tensor,
        *,
        bos_id: int,
        eos_id: int,
        max_len: int = 128,
    ) -> list[int]:
        self.eval()
        device = en_input_ids.device

        # 1. Encode source text once
        memory, memory_padding = self.encode(en_input_ids, en_attention_mask)

        # Start sequence with the BOS token
        generated = [bos_id]

        # 2. Autoregressive loop
        for _ in range(max_len):
            # Create a 2D tensor with batch dimension [1, current_sequence_length]
            tgt_ids = torch.tensor([generated], device=device, dtype=torch.long)

            # Pass to decoder (decoder automatically handles the causal tgt_mask)
            logits = self.decoder(
                tgt_ids,
                memory,
                memory_key_padding_mask=memory_padding
            )

            # Extract logits for the LAST predicted token position
            next_token_logits = logits[0, -1, :].clone()

            # Prevent the model from choosing its own PAD or BOS token during generation
            next_token_logits[self.pad_id] = float('-inf')
            next_token_logits[bos_id] = float('-inf')

            # Select the highest probability token ID
            next_id = int(next_token_logits.argmax(dim=-1).item())
            generated.append(next_id)

            # Stop if the model predicts the End of Sequence token
            if next_id == eos_id:
                break

        # 3. Strip special framing tokens so the tokenizer receives clean text IDs
        clean_generated = [tok for tok in generated if tok not in (bos_id, eos_id, self.pad_id)]
        return clean_generated

In [ ]:
!pip install -q onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 37.9 MB/s eta 0:00:00


In [ ]:
import os
import sys

# Safety path alignment
if os.getcwd() != "/content/lang-translator":
    %cd /content/lang-translator

print("🚀 STARTING MACHINE LEARNING PIPELINE IN SEQUENCE...\n")

def run_step(step_num, title, command):
    print(f"[{step_num}] {title}...")

    # Run the terminal command and grab the text list output
    output = get_ipython().getoutput(command)
    output_text = "\n".join(output)

    # Check if the terminal printed a fatal Python error
    if "Error:" in output_text or "Traceback" in output_text or "fatal:" in output_text:
        print(f"\n❌ {title} failed! Full Error Log Below:\n" + "-"*50)
        print(output_text)
        print("-"*50)
        raise RuntimeError(f"Pipeline halted at Step {step_num}")

    # If it worked, print whatever logs it normally prints
    print(output_text)
    print(f"✅ Step {step_num} Complete.\n")

# --- Run the steps sequentially ---
#run_step("1/7", "Downloading encoder", "python -m src.data.download_encoder")
#run_step("2/7", "Preparing corpus", "python -m src.data.prepare_corpus --input data/raw/sample_en_zanx_pairs.tsv")
#run_step("3/7", "Building zanX tokenizer", "python -m src.data.build_zanx_tokenizer --input data/processed/train.jsonl")
#run_step("4/7", "Training model on T4 GPU", "python -m src.train.train --config configs/dev.yaml")
#run_step("5/7", "Evaluating model", "python -m src.train.evaluate --config configs/dev.yaml --split test")
#run_step("6/7", "Exporting model to ONNX format", "python -m src.export.export_onnx --config configs/dev.yaml")
run_step("7/7", "Running inference smoke test", "python -m src.infer.onnx_runtime_infer --bundle artifacts/releases/v2 --text 'Hello world'")

print("🎉 [DONE] Entire pipeline completed successfully inside Google Colab!")


🚀 STARTING MACHINE LEARNING PIPELINE IN SEQUENCE...

[7/7] Running inference smoke test...

❌ Running inference smoke test failed! Full Error Log Below:
--------------------------------------------------
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/lang-translator/src/infer/onnx_runtime_infer.py", line 82, in <module>
    main()
  File "/content/lang-translator/src/infer/onnx_runtime_infer.py", line 78, in main
    print(translator.translate(args.text))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/lang-translator/src/infer/onnx_runtime_infer.py", line 55, in translate
    logits = self.decoder.run(
             ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py", line 320, in run
    return self._sess.run(output_names, input_feed, run_options)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

RuntimeError: Pipeline halted at Step 7/7

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q netron

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 21.4 MB/s eta 0:00:00


In [ ]:
from google.colab import auth
from google.auth import default
import os

print("Authenticating via native Google API...")
# This uses a standard, lightweight authentication loop
auth.authenticate_user()
creds, _ = default()

# Since we can't create a direct folder link, we will save files locally first,
# then we will use a python script at the end to copy them to your Drive.
os.makedirs("/content/lang-translator/artifacts", exist_ok=True)
print("✅ API Authentication Complete. Local workspace prepared.")


Authenticating via native Google API...


MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
import sys
import shutil
from google.colab import userdata

# 1. CLEAN RE-CLONE
TOKEN = userdata.get('GITHUB_TOKEN')
USER = "nimatamang"
REPO = "lang-translator"

if os.path.exists(REPO):
    shutil.rmtree(REPO)

print("🔄 Local storage was cleared. Re-cloning repository from GitHub...")
!git clone https://{USER}:{TOKEN}@://github.com{USER}/{REPO}.git

%cd {REPO}

# 2. ENVIRONMENT INSTALLATION
print("\n📦 Installing project packages...")
!pip install -q -r requirements.txt
!pip install -q netron

# 3. APPLY THE INFERENCE CODE FIX AUTOMATICALLY
# This rewrites the inference script to fix the 'tgt_mask' error without manual editing
infer_file_path = "src/infer/onnx_runtime_infer.py"

fixed_code = """from __future__ import annotations
import argparse
import json
from pathlib import Path
import numpy as np
import onnxruntime as ort
from transformers import BertTokenizer
from src.config import resolve_path
from src.data.zanx_tokenizer import ZanxTokenizer

class OnnxTranslator:
    def __init__(self, bundle_dir: str | Path) -> None:
        bundle = Path(bundle_dir)
        with (bundle / "manifest.json").open(encoding="utf-8") as handle:
            manifest = json.load(handle)

        self.zanx_tokenizer = ZanxTokenizer.load(bundle)
        self.en_tokenizer = BertTokenizer.from_pretrained(bundle / "en_tokenizer")
        self.bos_id = self.zanx_tokenizer.bos_id
        self.eos_id = self.zanx_tokenizer.eos_id

        encoder_path = bundle / manifest["encoder_onnx"]
        decoder_path = bundle / manifest["decoder_onnx"]
        self.encoder = ort.InferenceSession(str(encoder_path), providers=["CPUExecutionProvider"])
        self.decoder = ort.InferenceSession(str(decoder_path), providers=["CPUExecutionProvider"])

    def translate(self, text: str, *, max_len: int = 128) -> str:
        encoding = self.en_tokenizer(text, return_tensors="np", padding=True, truncation=True, max_length=max_len)
        input_ids = encoding["input_ids"].astype(np.int64)
        attention_mask = encoding["attention_mask"].astype(np.int64)
        memory = self.encoder.run(None, {"input_ids": input_ids, "attention_mask": attention_mask})[0]
        memory_mask = attention_mask == 0

        generated = [self.bos_id]
        for _ in range(max_len):
            tgt_ids = np.array([generated], dtype=np.int64)
            logits = self.decoder.run(
                None,
                {
                    "tgt_ids": tgt_ids,
                    "memory": memory,
                    "memory_key_padding_mask": memory_mask,
                },
            )[0]
            next_id = int(np.argmax(logits[:, -1, :], axis=-1)[0])
            generated.append(next_id)
            if next_id == self.eos_id:
                break
        return self.zanx_tokenizer.decode(generated)

def main() -> None:
    parser = argparse.ArgumentParser(description="Run ONNX English to zanX inference.")
    parser.add_argument("--bundle", default="artifacts/releases/v2")
    parser.add_argument("--text", required=True)
    args = parser.parse_args()
    translator = OnnxTranslator(resolve_path(args.bundle))
    print(translator.translate(args.text))

if __name__ == "__main__":
    main()
"""

with open(infer_file_path, "w", encoding="utf-8") as f:
    f.write(fixed_code)
print(f"🛠️ Applied the 'tgt_mask' bug fix directly to {infer_file_path}")

# 4. RUN ALL PIPELINE STEPS IN SEQUENCE
def run_pipeline_step(step_num, title, command):
    print(f"\n[{step_num}] {title}...")
    output = get_ipython().getoutput(command)
    output_text = "\n".join(output)
    if "Error:" in output_text or "Traceback" in output_text or "fatal:" in output_text:
        print(f"❌ {title} failed! Error Log Below:\\n" + "-"*50)
        print(output_text)
        print("-"*50)
        raise RuntimeError(f"Pipeline crashed at Step {step_num}")
    print(output_text)

run_pipeline_step("1/7", "Downloading encoder", "python -m src.data.download_encoder")
run_pipeline_step("2/7", "Preparing corpus", "python -m src.data.prepare_corpus --input data/raw/sample_en_zanx_pairs.tsv")
run_pipeline_step("3/7", "Building zanX tokenizer", "python -m src.data.build_zanx_tokenizer --input data/processed/train.jsonl")
run_pipeline_step("4/7", "Training model on T4 GPU", "python -m src.train.train --config configs/dev.yaml")
run_pipeline_step("5/7", "Evaluating model", "python -m src.train.evaluate --config configs/dev.yaml --split test")
run_pipeline_step("6/7", "Exporting model to ONNX format", "python -m src.export.export_onnx --config configs/dev.yaml")
run_pipeline_step("7/7", "Running inference smoke test", "python -m src.infer.onnx_runtime_infer --bundle artifacts/releases/v2 --text 'Hello world'")

print("\n🎉 [SUCCESS] Pipeline completed safely! The code error has been permanently patched.")


🔄 Local storage was cleared. Re-cloning repository from GitHub...
Cloning into 'lang-translator'...
fatal: unable to access 'https://://github.comnimatamang/lang-translator.git/': URL using bad/illegal format or missing URL
[Errno 2] No such file or directory: 'lang-translator'
/content

📦 Installing project packages...
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


FileNotFoundError: [Errno 2] No such file or directory: 'src/infer/onnx_runtime_infer.py'

In [ ]:
!python -m src.infer.onnx_runtime_infer --bundle artifacts/releases/v2 --text "Hello world"

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/lang-translator/src/infer/onnx_runtime_infer.py", line 75, in <module>
    main()
  File "/content/lang-translator/src/infer/onnx_runtime_infer.py", line 72, in main
    print(translator.translate(args.text))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/lang-translator/src/infer/onnx_runtime_infer.py", line 50, in translate
    logits = self.decoder.run(
             ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py", line 320, in run
    return self._sess.run(output_names, input_feed, run_options)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
onnxruntime.capi.onnxruntime_pybind11_state.InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Invalid input name: tgt_mask
